<a href="https://colab.research.google.com/github/busycaesar/LoRA/blob/Master/google-gemma/gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Kaggle Authentication
To download the Gemma model checkpoints, we authenticate using our Kaggle API credentials. If you do not have your credentials yet, you can create and download your API token from your [Kaggle Account](https://www.kaggle.com). Once downloaded, store them securely in Colab's secrets manager.

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### Environment Setup & Installation

In [ ]:
%pip install -U -q keras-hub keras

### Configure Keras Backend
We are using a tool called **JAX** as our backend because it makes training large AI models run much faster. We also set a memory limit (`XLA_PYTHON_CLIENT_MEM_FRACTION = "1.00"`). This tells JAX to reserve 100% of the graphics card (GPU) memory upfront.

In [ ]:
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

### Import Frameworks

In [ ]:
import keras
import keras_hub

### Load Gemma 3

In [ ]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")

### Define Prompt Template
This is the template that we wil use to send the prompt to the LLM. We will apprend the prompt into the `instruction` and leave the `response` empty for the LLM to fill.

In [ ]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

### Initial Response

In [ ]:
prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)

print(gemma_lm.generate(prompt, max_length=256))

### Download Training Dataset
We fetch the Databricks Dolly 15k dataset, which is a collection of high-quality, human-generated instruction/response pairs.

In [ ]:
!wget -O databricks-dolly-15k.jsonl https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl

### Parse and Format Dataset
We parse the first 1000 examples, filtering out items that require background context to keep our instruction-response fine-tuning setup simple.

In [ ]:
import json

prompts = []
responses = []
line_count = 0

with open("databricks-dolly-15k.jsonl") as file:
    for line in file:
        if line_count >= 1000:
            break  # Limit the training examples, to reduce execution time.

        examples = json.loads(line)
        # Filter out examples with context, to keep it simple.
        if examples["context"]:
            continue
        # Format data into prompts and response lists.
        prompts.append(examples["instruction"])
        responses.append(examples["response"])

        line_count += 1

features = {
    "prompts": prompts,
    "responses": responses
}

### Verify Parsed Data

In [ ]:
print(len(features['prompts']))
print(features['prompts'][0])
print(features['responses'][0])

### Summary of the LLM

In [ ]:
gemma_lm.summary()

### Enable Low-Rank Adaptation (LoRA)

In [ ]:
gemma_lm.backbone.enable_lora(rank=4)

### Summary of the LLM

In [ ]:
gemma_lm.summary()

### Compile Model
We prepare the model for training by applying standard settings (an optimizer and a loss function) that will help it learn from our dataset.

In [ ]:
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)

optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

### Model Training

In [ ]:
gemma_lm.fit(features, epochs=1, batch_size=1)

### Evaluate Fine-Tuned Model

In [ ]:
prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)

print(gemma_lm.generate(prompt, max_length=256))